# H20a: Testing whether a sampled Newton-alignment advantage compounds over training

**Paired implementation:** `run_experiment.py`

## Truthful framing
This notebook is a presentation layer for the exact computation performed by the paired script. It does **not** re-implement the experiment logic independently. Instead, it imports `run_experiment.py`, executes `run_experiment()`, and builds tables/figures from the returned structured results.

## Hypothesis under test
In this toy deep-linear setting, Muon often shows a positive **sampled** Newton-alignment advantage over NormSGD. The question here is whether that sampled directional edge is associated with a meaningful compounding improvement in the loss ratio over training.

## Primary test used in this first pass
We examine a matched sampled-state regression:

\[
\log \left( 	ext{loss}_{	ext{NormSGD}} / 	ext{loss}_{	ext{Muon}} ight)
\quad 	ext{vs.} \quad
	ext{cumulative sampled cosine advantage}.
\]

A positive relationship would be at least *consistent* with a compounding story; a non-positive or negligible relationship would not support that claim under the current configuration.


## Scope and limits

- **Toy problem:** 2-layer, 4x4 deep linear regression with fixed synthetic data per seed.
- **Sparse Hessian sampling:** the Newton reference is computed only at explicit sample steps, not every step.
- **Sampled proxy metric:** the cumulative cosine quantity is a running sum over those sampled steps only.
- **Cross-trajectory comparison:** the Newton direction is computed at Muon's state, so the NormSGD comparison is not a same-point control.

This notebook therefore supports an honest *proxy analysis*, not a definitive mechanistic resolution of any broader Muon paradox.


In [ ]:
import importlib.util
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

RELATIVE_EXPERIMENT_DIR = Path('experiments/Tier_1_Core_Mechanism_Tests/H20a_COSINE_COMPOUNDING_TRAJECTORY')


def resolve_experiment_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / RELATIVE_EXPERIMENT_DIR]
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / RELATIVE_EXPERIMENT_DIR)
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'run_experiment.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate the H20a experiment directory containing run_experiment.py. '
        f'Tried from cwd={cwd}.'
    )


def load_experiment_module(script_path: Path):
    spec = importlib.util.spec_from_file_location('h20a_run_experiment', script_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


EXPERIMENT_DIR = resolve_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
experiment_module = load_experiment_module(SCRIPT_PATH)

print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Script path:         {SCRIPT_PATH}')
print(f'Python:              {sys.version.split()[0]}')
print(f'Platform:            {platform.platform()}')
print(f'NumPy:               {np.__version__}')
print(f'Matplotlib:          {plt.matplotlib.__version__}')
print(f'Pandas available:    {pd is not None}')


## Reproducibility, runtime, and configuration

The cell below is the only place where the experiment is executed. Everything else in the notebook reads from the returned `results` dictionary.


In [ ]:
t0 = time.time()
results = experiment_module.run_experiment(verbose=False)
notebook_elapsed = time.time() - t0

config = results['config']
lr_search = results['lr_search']

print('Execution summary')
print('-' * 80)
print(f"Experiment ID:                     {results['experiment_id']}")
print(f"Notebook-side wall time:           {notebook_elapsed:.3f} s")
print(f"Reported run_experiment() time:    {results['runtime_seconds']:.3f} s")
print(f"Seeds:                              {results['seeds']}")
print(f"Network:                            {config['n_layers']}-layer, dim={config['dim']}, params={config['n_params']}")
print(f"Training steps:                     {config['num_steps']}")
print(f"Batch size:                         {config['batch_size']}")
print(f"Checkpoint steps (1-based):         {config['checkpoints_1based_post_update']}")
print(f"Hessian sample steps (0-based):     {config['hessian_sample_steps_0based_pre_update']}")
print(f"Muon LR grid size:                  {len(lr_search['muon']['grid'])}")
print(f"NormSGD LR grid size:               {len(lr_search['normsgd']['grid'])}")
print(f"Chosen Muon LR:                     {lr_search['muon']['selected_best_lr']:.12f}")
print(f"Chosen NormSGD LR:                  {lr_search['normsgd']['selected_best_lr']:.12f}")
print('Sampled cumulative cosine definition:')
print(f"  {config['sampled_cumulative_cosine_definition']}")


**Interpretation.** The notebook now depends on the script as a single source of truth. The explicit sample schedule and the definition of the cumulative cosine proxy are logged before any interpretation is attempted.


In [ ]:
sample = results['aggregates']['sample_states']
check = results['aggregates']['checkpoints']
per_seed = results['per_seed']

sample_steps = np.array(sample['steps_0based_pre_update'])
checkpoint_steps = np.array(check['steps_1based_post_update'])

sample_mean_cos_adv = np.array(sample['cosine_advantage']['mean'])
sample_ci_cos_adv = np.array(sample['cosine_advantage']['ci95'])
sample_mean_cumul = np.array(sample['cumulative_sampled_cosine_advantage_pre_update']['mean'])
sample_mean_ratio = np.array(sample['loss_ratio']['mean'])
sample_ci_ratio = np.array(sample['loss_ratio']['ci95'])
sample_log_mean_ratio = np.array(sample['log_mean_loss_ratio'])

checkpoint_mean_cumul = np.array(check['cumulative_sampled_cosine_advantage']['mean'])
checkpoint_mean_ratio = np.array(check['loss_ratio']['mean'])
checkpoint_ci_ratio = np.array(check['loss_ratio']['ci95'])
checkpoint_log_mean_ratio = np.array(check['log_mean_loss_ratio'])

fit_sampled = results['fits']['sampled_log_mean_loss_ratio_vs_mean_cumulative_sampled_cosine']
fit_checkpoint_time = results['fits']['checkpoint_log_mean_loss_ratio_vs_step']


def show_records(records, precision=6):
    if pd is not None:
        table = pd.DataFrame.from_records(records).round(precision)
        print(table.to_string(index=False))
        return table
    for row in records:
        print(row)
    return records


## Summary tables

The tables below separate the two reporting schedules used by the script:

1. **Matched sampled states** (0-based, pre-update) used for the main proxy fit.
2. **Original checkpoints** (1-based, post-update) used for the training-loss summaries.


In [ ]:
config_records = [
    {'quantity': 'num_seeds', 'value': config['num_seeds']},
    {'quantity': 'num_steps', 'value': config['num_steps']},
    {'quantity': 'hessian_sample_every', 'value': config['hessian_sample_every']},
    {'quantity': 'muon_best_lr', 'value': lr_search['muon']['selected_best_lr']},
    {'quantity': 'normsgd_best_lr', 'value': lr_search['normsgd']['selected_best_lr']},
    {'quantity': 'sampled_fit_slope', 'value': fit_sampled['slope']},
    {'quantity': 'sampled_fit_r2', 'value': fit_sampled['r2']},
    {'quantity': 'final_checkpoint_mean_ratio', 'value': checkpoint_mean_ratio[-1]},
    {'quantity': 'final_parameter_divergence_mean', 'value': results['aggregates']['final_parameter_divergence']['mean']},
]
print('Key run summary')
print('-' * 80)
show_records(config_records, precision=6)

sampled_records = []
for i, step in enumerate(sample_steps):
    sampled_records.append({
        'sample_step_pre_update': int(step),
        'mean_cosine_advantage': sample_mean_cos_adv[i],
        'ci95_cosine_advantage': sample_ci_cos_adv[i],
        'mean_cumulative_sampled_cosine': sample_mean_cumul[i],
        'mean_loss_ratio': sample_mean_ratio[i],
        'ci95_loss_ratio': sample_ci_ratio[i],
        'log_mean_loss_ratio': sample_log_mean_ratio[i],
    })
print()
print('Matched sampled-state summary')
print('-' * 80)
show_records(sampled_records, precision=6)

checkpoint_records = []
for i, step in enumerate(checkpoint_steps):
    checkpoint_records.append({
        'checkpoint_step_post_update': int(step),
        'mean_cumulative_sampled_cosine': checkpoint_mean_cumul[i],
        'mean_loss_ratio': checkpoint_mean_ratio[i],
        'ci95_loss_ratio': checkpoint_ci_ratio[i],
        'log_mean_loss_ratio': checkpoint_log_mean_ratio[i],
    })
print()
print('Checkpoint summary')
print('-' * 80)
show_records(checkpoint_records, precision=6)


**Interpretation.** The main empirical pattern is already visible in the tables: a positive sampled cosine advantage exists, but the mean loss ratio moves back toward approximately **1x** after the early part of training. That behavior does not look like strong sustained compounding under the current setup.


## Figure 1 — Sampled cosine advantage versus sample step

This figure shows the sparse Newton-alignment advantage that is actually measured by the script. Faint lines are individual seeds; the bold curve is the mean with a 95% CI band.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for seed_result in per_seed:
    ax.plot(
        sample_steps,
        seed_result['sampled_cosine_advantage'],
        color='tab:blue',
        alpha=0.25,
        marker='o',
        linewidth=1.0,
    )
ax.plot(sample_steps, sample_mean_cos_adv, color='black', marker='o', linewidth=2.5, label='Mean')
ax.fill_between(
    sample_steps,
    sample_mean_cos_adv - sample_ci_cos_adv,
    sample_mean_cos_adv + sample_ci_cos_adv,
    color='black',
    alpha=0.15,
    label='Mean ± 95% CI',
)
ax.axhline(0.0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Hessian sample step (0-based, pre-update)')
ax.set_ylabel('Sampled cosine advantage')
ax.set_title('Muon minus NormSGD Newton alignment at sampled states')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


**Interpretation.** The sampled cosine advantage is mostly positive and tends to be larger later in training. That is consistent with Muon often being better aligned with the local Newton direction at the sampled states. However, directional advantage alone does not establish compounding in loss.


## Figure 2 — Loss-ratio trajectories

The left panel uses the matched sampled states that feed the main proxy regression. The right panel uses the original post-update checkpoints retained from the first version of the experiment.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)

for seed_result in per_seed:
    axes[0].plot(
        sample_steps,
        seed_result['sampled_loss_ratio'],
        color='tab:orange',
        alpha=0.25,
        marker='o',
        linewidth=1.0,
    )
axes[0].plot(sample_steps, sample_mean_ratio, color='black', marker='o', linewidth=2.5, label='Mean')
axes[0].fill_between(
    sample_steps,
    sample_mean_ratio - sample_ci_ratio,
    sample_mean_ratio + sample_ci_ratio,
    color='black',
    alpha=0.15,
    label='Mean ± 95% CI',
)
axes[0].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[0].set_xlabel('Sample step (0-based, pre-update)')
axes[0].set_ylabel('Loss ratio = loss_NormSGD / loss_Muon')
axes[0].set_title('Matched sampled states')
axes[0].legend(loc='best')

for seed_result in per_seed:
    axes[1].plot(
        checkpoint_steps,
        seed_result['checkpoint_loss_ratio'],
        color='tab:green',
        alpha=0.25,
        marker='o',
        linewidth=1.0,
    )
axes[1].plot(checkpoint_steps, checkpoint_mean_ratio, color='black', marker='o', linewidth=2.5, label='Mean')
axes[1].fill_between(
    checkpoint_steps,
    checkpoint_mean_ratio - checkpoint_ci_ratio,
    checkpoint_mean_ratio + checkpoint_ci_ratio,
    color='black',
    alpha=0.15,
    label='Mean ± 95% CI',
)
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_xlabel('Checkpoint step (1-based, post-update)')
axes[1].set_ylabel('Loss ratio = loss_NormSGD / loss_Muon')
axes[1].set_title('Original checkpoint reporting')
axes[1].legend(loc='best')

plt.tight_layout()
plt.show()


**Interpretation.** Muon has an early advantage in the loss ratio, but the gap contracts strongly and is essentially gone by the later checkpoints in the mean trajectory. This is the main reason the current notebook should not claim established geometric compounding.


## Figure 3 — Proxy fit: log(loss ratio) versus cumulative sampled cosine

This is the main fit used in the first-pass cleanup. The black points are the mean sampled states, the red line is the fit used by the script, and the faint gray points show individual-seed sampled states for context.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for seed_result in per_seed:
    x_seed = np.array(seed_result['sampled_cumulative_cosine_advantage_pre_update'])
    y_seed = np.log(np.maximum(np.array(seed_result['sampled_loss_ratio']), 1e-30))
    ax.scatter(x_seed, y_seed, color='gray', alpha=0.25, s=28)

x_mean = sample_mean_cumul
y_mean = sample_log_mean_ratio
x_fit = np.linspace(float(np.min(x_mean)), float(np.max(x_mean)), 200)
y_fit = fit_sampled['slope'] * x_fit + fit_sampled['intercept']

ax.scatter(x_mean, y_mean, color='black', s=60, label='Mean sampled states')
for step, x_val, y_val in zip(sample_steps, x_mean, y_mean):
    ax.annotate(str(int(step)), (x_val, y_val), textcoords='offset points', xytext=(5, 4), fontsize=8)
ax.plot(
    x_fit,
    y_fit,
    color='tab:red',
    linewidth=2.0,
    label=f"Mean fit: slope={fit_sampled['slope']:.4f}, R^2={fit_sampled['r2']:.3f}",
)
ax.set_xlabel('Mean cumulative sampled cosine advantage (pre-update matched states)')
ax.set_ylabel('log(mean loss ratio)')
ax.set_title('Matched sampled-state proxy fit')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


**Interpretation.** In the current verified run, the fitted slope is non-positive and the relationship is weak. Under this proxy analysis, the present implementation does **not** support a geometric-compounding claim.


## Final verdict and limitations

The goal of this notebook is not to prove a grand mechanism; it is to present the current toy computation honestly and transparently.


In [ ]:
print('Verdict')
print('-' * 80)
print(f"Status:  {results['verdict']['status']}")
print(f"Message: {results['verdict']['message']}")
print()
print('Fit diagnostics')
print('-' * 80)
print(f"Matched sampled-state slope:   {fit_sampled['slope']:.8f}")
print(f"Matched sampled-state R^2:     {fit_sampled['r2']:.6f}")
print(f"Checkpoint time slope:         {fit_checkpoint_time['slope']:.8f}")
print(f"Checkpoint time R^2:           {fit_checkpoint_time['r2']:.6f}")
print()
print('Final parameter divergence summary')
print('-' * 80)
fpd = results['aggregates']['final_parameter_divergence']
print(f"Mean final ||theta_muon - theta_norm||: {fpd['mean']:.6f}")
print(f"95% CI:                                {fpd['ci95']:.6f}")
print()
print('Script-reported methodological notes')
print('-' * 80)
for note in results['notes']:
    print(f'- {note}')


### Honest conclusion

- The current experiment **does** show a positive sampled Newton-alignment advantage for Muon at many sampled states.
- The current experiment **does not** show a sustained loss-ratio separation consistent with strong geometric compounding.
- The matched sampled-state fit is weak/non-positive in the verified default run.
- Therefore this first-pass pair should be read as a **truthful proxy analysis in a toy setting**, not as evidence that the compounding mechanism has already been established or that the broader paradox has been resolved.

### Recommended next verification after this cleanup

1. Execute the notebook end-to-end inside Jupyter and confirm that the figures render cleanly.
2. Compare the notebook tables directly against the CLI output from `run_experiment.py`.
3. If a stronger mechanistic claim is still desired, add a same-point control metric in a later pass rather than strengthening the current rhetoric.
